In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression, Lasso, LassoCV
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import norm
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from scipy.stats import randint, uniform
from scipy.stats import pearsonr

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Importing all running csvs
a = pd.read_csv('/kaggle/input/long-distance-running-dataset/run_ww_2019_w.csv')
b = pd.read_csv('/kaggle/input/long-distance-running-dataset/run_ww_2019_m.csv')
c = pd.read_csv('/kaggle/input/long-distance-running-dataset/run_ww_2019_d.csv')
d = pd.read_csv('/kaggle/input/long-distance-running-dataset/run_ww_2020_w.csv')
e = pd.read_csv('/kaggle/input/long-distance-running-dataset/run_ww_2020_d.csv')
f = pd.read_csv('/kaggle/input/long-distance-running-dataset/run_ww_2019_q.csv')
g = pd.read_csv('/kaggle/input/long-distance-running-dataset/run_ww_2020_m.csv')
h = pd.read_csv('/kaggle/input/long-distance-running-dataset/run_ww_2020_q.csv')

In [ ]:
original = pd.concat([a,b,c,d,e,f,g,h])

In [ ]:
df = original

## Initial Exploration

In [ ]:
df.describe()

In [ ]:
df.columns

In [ ]:
df.head()

In [ ]:
print(df['age_group'].unique())

In [ ]:
print('Distance Min: ', df['distance'].min())
print('Distance Max: ',df['distance'].max())

print('Duration Min: ',df['distance'].min())
print('Duration Min: ',df['distance'].max())

In [ ]:
# Duplicate check
df.duplicated().sum()

## Data Cleaning

In [ ]:
df = df.drop_duplicates()
print(df.count())

In [ ]:
# Removing 0 distance + duration
df = df[df['distance'] != 0]
df = df[df['duration'] != 0]

In [ ]:
# Removing unhelpful columns
df = df.drop(['Unnamed: 0', 'datetime', 'athlete', 'country', 'major'], axis = 1)

In [ ]:
df.isnull().sum() # No Nulls

## Column Relationships

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df,
    x='distance',
    y='duration',
    # estimator=np.mean, # Or np.sum, np.median, etc.
    # errorbar=None, # Remove confidence intervals if not desired
)
plt.xlabel('Distance')
plt.ylabel('Duration')
plt.title('Distance by Duration')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
correlation, p_value = pearsonr(df['distance'], df['duration'])

print(f"Pearson Correlation Coefficient: {correlation}")
print(f"P-value: {p_value}")

In [ ]:
display(pd.DataFrame(df['age_group'].value_counts()))

plt.figure(figsize = (10, 8))
sns.countplot(x = df['age_group'])
plt.title('Number of runs based on the age')
plt.xlabel('Age')
plt.show()


In [ ]:
# df['age_group'] = pd.Categorical(df['age_group'], ordered=True)


# 3. Create the box plot
plt.figure(figsize=(10, 7)) # Adjust figure size for better readability

sns.boxplot(data=df, x='age_group', y='duration', palette='viridis') # 'viridis' is a nice colormap

# Add titles and labels
plt.title('Duration Distribution by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Duration of Run')

In [ ]:
df['gender'] = pd.Categorical(df['gender'], ordered=True)


# 3. Create the box plot
plt.figure(figsize=(10, 7)) # Adjust figure size for better readability

sns.boxplot(data=df, x='gender', y='duration', palette='viridis') # 'viridis' is a nice colormap

# Add titles and labels
plt.title('Duration Distribution by Gender')
plt.xlabel('Gender')
plt.ylabel('Duration of Run')

## Final dataset

In [ ]:
df = df[(df['distance'] <= 5)]

In [ ]:
# Q1 = df['duration'].quantile(0.25)
# Q3 = df['duration'].quantile(0.75)
# IQR = Q3 - Q1

# print(f"Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")

# df = df[(df['duration'] >= Q1) & (df['duration'] <= Q3)]

In [ ]:
correlation, p_value = pearsonr(df['distance'], df['duration'])

print(f"Pearson Correlation Coefficient: {correlation}")
print(f"P-value: {p_value}")

In [ ]:
df.head()

In [ ]:
age_order = [['18 - 34', '35 - 54', '55 +']] # 2d Array
age_encoder = OrdinalEncoder(categories=age_order)
df['age_group'] = age_encoder.fit_transform(df[['age_group']])


g_encoder = OrdinalEncoder()
df['gender'] = g_encoder.fit_transform(df[['gender']])


scaler = StandardScaler()
# Select only numerical columns for scaling
numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns
# Apply standard scaling
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
# Print the scaled DataFrame
df.head()

In [ ]:
dff = df

In [ ]:
X = dff.drop(['duration'], axis=1)
y = dff['duration']
# y_log = np.log1p(y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
print((y <= 0).sum())
y = y[y > 0]
print((y <= 0).sum())

In [ ]:
y = y.dropna()

## Models

In [ ]:
# Try log(y) with LR
# Try GB

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

y_train_pred = model.predict(X_train)
y_pred = model.predict(X_test)

mse_train = mean_squared_error(y_train, y_train_pred)
r2_train = r2_score(y_train, y_train_pred)

print('-'*50)
print('Train metrics')
print(f"Mean Squared Error (MSE): {mse_train:.2f}")
print(f"Root Mean Squared Error (RMSE): {np.sqrt(mse_train):.2f}")
print(f"R-squared (R²): {r2_train:.2f}")

print('-'*50)
print('Test metrics')
mse_test = mean_squared_error(y_test, y_pred)
r2_test = r2_score(y_test, y_pred)

print(f"Mean Squared Error (MSE): {mse_test:.2f}")
print(f"Root Mean Squared Error (RMSE): {np.sqrt(mse_test):.2f}")
print(f"R-squared (R²): {r2_test:.2f}")

In [ ]:
residuals = y_test - y_pred
plt.scatter(residuals,y_pred)

plt.show()

In [ ]:
correlation, p_value = pearsonr(df['distance'], df['duration'])

print(f"Pearson Correlation Coefficient: {correlation}")
print(f"P-value: {p_value}")

In [ ]:
model = Lasso(alpha=0.02)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Model Intercept (β0): {model.intercept_:.2f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {np.sqrt(mse):.2f}")
print(f"R-squared (R²): {r2:.2f}")

In [ ]:
model = DecisionTreeRegressor(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

# Evaluate
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
print("Mean Squared Error:", mse)
print(f"Root Mean Squared Error (RMSE): {np.sqrt(mse):.2f}")
print(f"R-squared (R²): {r2:.2f}")

In [ ]:
residuals = y_test - y_pred
plt.scatter(residuals,y_pred)

plt.show()

correlation, p_value = pearsonr(df['distance'], df['duration'])

print(f"Pearson Correlation Coefficient: {correlation}")
print(f"P-value: {p_value}")

In [ ]:
model = KNeighborsRegressor(n_neighbors=5)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

# Evaluate
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
print("Mean Squared Error:", mse)
print(f"Root Mean Squared Error (RMSE): {np.sqrt(mse):.2f}")
print(f"R-squared (R²): {r2:.2f}")

In [ ]:
# rf = RandomForestRegressor(n_estimators=100, random_state=42)
# rf.fit(X_train, y_train)

# y_pred = rf.predict(X_test)

# # Evaluate
# r2 = r2_score(y_test, y_pred)
# mse = mean_squared_error(y_test, y_pred)
# print("Mean Squared Error:", mse)
# print(f"Root Mean Squared Error (RMSE): {np.sqrt(mse):.2f}")
# print(f"R-squared (R²): {r2:.2f}")

In [ ]:
model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=500, learning_rate=0.1,  enable_categorical=True)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_train_pred = model.predict(X_train)

mse_train = mean_squared_error(y_train, y_train_pred)
r2_train = r2_score(y_train, y_train_pred)

print('-'*50)
print('Train metrics')
print(f"Mean Squared Error (MSE): {mse_train:.2f}")
print(f"Root Mean Squared Error (RMSE): {np.sqrt(mse_train):.2f}")
print(f"R-squared (R²): {r2_train:.2f}")

print('-'*50)
print('Test metrics')
mse_test = mean_squared_error(y_test, y_pred)
r2_test = r2_score(y_test, y_pred)

print(f"Mean Squared Error (MSE): {mse_test:.2f}")
print(f"Root Mean Squared Error (RMSE): {np.sqrt(mse_test):.2f}")
print(f"R-squared (R²): {r2_test:.2f}")

In [ ]:
residuals = y_train - y_train_pred
plt.scatter(residuals,y_train_pred)

plt.show()

correlation, p_value = pearsonr(df['distance'], df['duration'])

print(f"Pearson Correlation Coefficient: {correlation}")
print(f"P-value: {p_value}")

In [ ]:
residuals = y_test - y_pred
plt.scatter(residuals,y_pred)

plt.show()

correlation, p_value = pearsonr(df['distance'], df['duration'])

print(f"Pearson Correlation Coefficient: {correlation}")
print(f"P-value: {p_value}")

In [ ]:
X.corr()

In [ ]:
X_test.corr()